In [ ]:
!pip install -q sentence-transformers datasets einops

In [ ]:
from datasets import load_dataset, Dataset
import random

raw_dataset = load_dataset("hsseinmz/arcd", split="train", trust_remote_code=True)

print(f"Raw samples: {len(raw_dataset)}")
print(f"Columns: {raw_dataset.column_names}")
print(f"\nSample: {raw_dataset[0]}")

In [ ]:
pairs = []

for item in raw_dataset:
    context = item["context"].strip()
    question = item.get("question", "").strip()

    if question and context:
        pairs.append({"anchor": question, "positive": context})

print(f"Extracted {len(pairs)} (question, context) pairs")
print(f"\nExample:")
print(f"  Q: {pairs[0]['anchor'][:80]}...")
print(f"  P: {pairs[0]['positive'][:80]}...")

random.seed(42)
random.shuffle(pairs)

split_idx = int(len(pairs) * 0.9)
train_pairs = pairs[:split_idx]
eval_pairs = pairs[split_idx:]

train_dataset = Dataset.from_list(train_pairs)
eval_dataset = Dataset.from_list(eval_pairs)

print(f"\nTrain: {len(train_dataset)}, Eval: {len(eval_dataset)}")

In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_ID = "nomic-ai/nomic-embed-text-v2-moe"

model = SentenceTransformer(MODEL_ID, trust_remote_code=True)

print(f"Embedding dim: {model.get_sentence_embedding_dimension()}")
print(f"Max seq length: {model.max_seq_length}")

test_emb = model.encode(["مرحبا بالعالم"], prompt_name="query")
print(f"Test embedding shape: {test_emb.shape}")

In [ ]:
from sentence_transformers import SentenceTransformerTrainer
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.evaluation import InformationRetrievalEvaluator

OUTPUT_DIR = "./nomic-embed-arabic-search"

train_loss = MultipleNegativesRankingLoss(model=model)

training_args = SentenceTransformerTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    fp16=True,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=100,
    logging_steps=25,
    save_total_limit=3,
    load_best_model_at_end=True,
    report_to=["tensorboard"],
)

print("Training config ready.")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  LR: {training_args.learning_rate}")

In [ ]:
queries = {}
corpus = {}
relevant_docs = {}

corpus_texts_seen = {}

for i, pair in enumerate(eval_pairs):
    qid = f"q{i}"
    queries[qid] = pair["anchor"]

    doc_text = pair["positive"]
    if doc_text not in corpus_texts_seen:
        cid = f"c{len(corpus_texts_seen)}"
        corpus_texts_seen[doc_text] = cid
        corpus[cid] = doc_text

    cid = corpus_texts_seen[doc_text]
    relevant_docs[qid] = {cid}

ir_evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name="arabic-search-eval",
    precision_recall_at_k=[1, 3, 5, 10],
    show_progress_bar=True,
)

print(f"IR Evaluator: {len(queries)} queries, {len(corpus)} documents")

In [ ]:
trainer = SentenceTransformerTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    loss=train_loss,
    evaluator=ir_evaluator,
)

print("Starting training...")
trainer.train()

In [ ]:
results = ir_evaluator(model)

print("\n" + "=" * 50)
print("Semantic Search Evaluation")
print("=" * 50)
for key, value in sorted(results.items()):
    if "precision" in key or "recall" in key or "mrr" in key or "ndcg" in key or "map" in key:
        print(f"  {key}: {value:.4f}")

In [ ]:
model.save_pretrained(f"{OUTPUT_DIR}/final")
print(f"Model saved to {OUTPUT_DIR}/final")

from google.colab import drive
drive.mount("/content/drive")

!cp -r {OUTPUT_DIR}/final /content/drive/MyDrive/nomic-embed-arabic-search
print("Model copied to Google Drive!")

In [ ]:
import numpy as np

documents = [
    "الذكاء الاصطناعي يغير عالم التكنولوجيا بشكل جذري ويؤثر على جميع القطاعات",
    "التعلم العميق هو فرع من التعلم الآلي يستخدم الشبكات العصبية العميقة",
    "معالجة اللغة الطبيعية تساعد الحواسيب على فهم وتحليل اللغة البشرية",
    "الطاقة الشمسية مصدر متجدد ونظيف للطاقة يستخدم في توليد الكهرباء",
    "التاريخ الإسلامي يمتد لأكثر من ألف وأربعمائة عام من الحضارة والعلم",
]

query = "ما هي الشبكات العصبية في الذكاء الاصطناعي"

doc_embeddings = model.encode(documents, prompt_name="passage", normalize_embeddings=True)
query_embedding = model.encode([query], prompt_name="query", normalize_embeddings=True)

similarities = np.dot(query_embedding, doc_embeddings.T)[0]

print(f"Query: {query}\n")
print("Results (ranked by similarity):")
for idx in np.argsort(similarities)[::-1]:
    print(f"  [{similarities[idx]:.4f}] {documents[idx]}")